Middleware provides a way tomore tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics and debugging.
- Transforming prompts, tool selection and output formatting.
- Adding retries, fallbacks, and early terminating logic.
- Applying rate limits, guardrails, and PII detection.

In [9]:
import os
from dotenv import load_dotenv 
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


### Summerization Middleware
Summarizes conversation history when token limits are approached.

This middleware monitors message token counts and automatically summarizes older messages when a threshold is reached, preserving recent messages and maintaining context continuity by ensuring AI/Tool message pairs remain together.

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

In [17]:
## Messagebase summarization
agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="groq:llama-3.1-8b-instant",
        trigger=("messages",6),
        keep=("messages",4)
        )]
)

In [15]:
### Run with thread id
config={"configurable":{"thread_id":"test_1"}}

In [18]:
#alternate test data
questions = [
    "what is 2+2",
    "what is 10*5",
    "what is OS in 10 words?",
    "what is 15-7"
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"messages: {response}")
    print(f"Messages: {len(response['messages'])}")

messages: {'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='ac121a00-c18e-4a48-94c9-581897d4fff3'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 41, 'total_tokens': 50, 'completion_time': 0.026128517, 'completion_tokens_details': None, 'prompt_time': 0.002898673, 'prompt_tokens_details': None, 'queue_time': 0.048563797, 'total_time': 0.02902719}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e7da3-c87c-7011-8a67-37e66122f1e8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 9, 'total_tokens': 50})]}
Messages: 2
messages: {'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='ac121a00-c18e-4a48-94c9-581897d4fff3'), AIMessage(co

## Based on Token Size

In [29]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) -> str:
    """Search hotels - return long response to use more tokens."""
    return f""" Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $150/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model="groq:llama-3.1-8b-instant",
        trigger=("tokens",440),
        keep=("tokens",300),
    )]
)

config= {"configurable":{"thread_id":"test_2"}}

# token counter {approximation}
def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4
# as 4 char = 1 token



In [30]:
# test run
cities = ["Paris","London","Tokyo","Dubai","Delhi","New York"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find the hotel in {city}")]},
        config
    )
    tokens = count_token(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~59 tokens, 4 messages
[HumanMessage(content='Find the hotel in Paris', additional_kwargs={}, response_metadata={}, id='a727b937-0bba-4fd0-95f9-d34a834a4eaf'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'zcghpynn4', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 227, 'total_tokens': 242, 'completion_time': 0.022867909, 'completion_tokens_details': None, 'prompt_time': 0.020778745, 'prompt_tokens_details': None, 'queue_time': 0.157417303, 'total_time': 0.043646654}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e7de8-29d9-7d53-84f8-9d3c2a724c34-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'zcghpynn4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

### now we will try trigger with `fraction`

In [35]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) -> str:
    """Search hotels - return long response to use more tokens."""
    return f""" Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $150/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model="groq:llama-3.1-8b-instant",
        trigger=("fraction",0.05),
        keep=("fraction",0.02),
    )]
)

config1= {"configurable":{"thread_id":"test_2"}}

# token counter {approximation}
def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4
# as 4 char = 1 token



In [36]:
# test run
cities = ["Paris","London","Tokyo","Dubai","Delhi","New York"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find the hotel in {city}")]},
        config1
    )
    tokens = count_token(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~83 tokens, 4 messages
[HumanMessage(content='Find the hotel in Paris', additional_kwargs={}, response_metadata={}, id='6495fad8-104f-441b-bbf0-f5b72ca483ac'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'zbn5fjgt5', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 227, 'total_tokens': 242, 'completion_time': 0.019975286, 'completion_tokens_details': None, 'prompt_time': 0.214743057, 'prompt_tokens_details': None, 'queue_time': 0.049743932, 'total_time': 0.234718343}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e7dec-137d-7172-8ee5-2ede52a72daa-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'zbn5fjgt5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat